# NB04a — Data Structures for Finance

**Role:** Compact-extension  
**Manual section:** 2.2.2.2 — Data structures  
**Prerequisites:** NB04

---

## Purpose of this notebook

This compact-extension notebook introduces data structures beyond flat tables: long vs wide formats, graphs as edge lists, and trees as decision structures. These formats appear in many financial contexts even though the core modelling pathway uses flat tables.

**Dataset:** `dataset_C_bank_attrition_core.csv` (illustrative) + inline toy data

## Section 0 — Why data structure matters

In finance, the same business phenomenon can be represented in very different ways. A customer relationship can be stored as a **table**, a payments network can be represented as a **graph**, and a rule-based approval logic can be represented as a **tree**. The key question is always: **what is the right structure for the question we want to answer?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from pathlib import Path
_candidates = [Path('data/processed'), Path('../data/processed')]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Cannot locate data/processed/. Launch from project root or notebooks/.')

import seaborn as sns

## Section 1 — Tables and units of analysis

Most of the course uses rectangular tables because they are the natural starting point for customer-level or contract-level analytics. But even inside tables, we still need to decide the **unit of analysis** carefully.

In [ ]:
df = pd.read_csv(DATA_DIR / 'dataset_C_bank_attrition_core.csv')
df.head()

In [ ]:
unit_examples = pd.DataFrame({
    'possible_unit_of_analysis': ['customer', 'customer-month', 'transaction', 'loan', 'claim'],
    'typical_question': [
        'Who is likely to churn?',
        'How does behaviour evolve month by month?',
        'Which transactions look abnormal?',
        'Which loans may default?',
        'Which claims deserve deeper review?'
    ]
})
unit_examples

## Section 2 — Long vs. wide representations

The same information may appear in **wide form** (one row, many month columns) or **long form** (one row per month). Long form is often better for time-based grouping, plotting and aggregation.

In [ ]:
monthly_balance = pd.DataFrame({
    'customer_id': [101, 102, 103],
    'balance_jan': [1200, 5400, 300],
    'balance_feb': [1150, 5600, 450],
    'balance_mar': [1300, 5700, 500],
})
monthly_balance

In [ ]:
long_balance = monthly_balance.melt(id_vars='customer_id', var_name='month', value_name='balance')
long_balance['month'] = long_balance['month'].str.replace('balance_', '', regex=False)
long_balance

In [ ]:
summary_long = long_balance.groupby('month', as_index=False)['balance'].mean()
summary_long

**Diagnose → transform → validate → justify**

- **Diagnose:** the wide table is easy to read for a few customers but becomes awkward for repeated analysis.
- **Transform:** use a long representation.
- **Validate:** check that row counts and totals still make sense.
- **Justify:** long form is usually better for temporal analytics and grouped summaries.

## Section 3 — Graphs as relationship structures

Some finance problems are naturally relational: money flows between accounts, firms share counterparties, and customers interact in payment networks. A graph is often stored as an **edge list**: source, target, amount.

In [ ]:
payments = pd.DataFrame({
    'from_account': ['A', 'A', 'B', 'C', 'D', 'E'],
    'to_account':   ['B', 'C', 'D', 'D', 'E', 'A'],
    'amount':       [500, 200, 750, 300, 150, 400]
})
payments

In [ ]:
accounts = sorted(set(payments['from_account']).union(payments['to_account']))
adj = pd.crosstab(payments['from_account'], payments['to_account'], values=payments['amount'], aggfunc='sum').reindex(index=accounts, columns=accounts).fillna(0)
adj

In [ ]:
plt.figure(figsize=(5,4))
sns.heatmap(adj, annot=True, fmt='.0f', cmap='Blues')
plt.title('Mini payment network as adjacency matrix')
plt.show()

## Section 4 — Trees as decision structures

A tree stores a sequence of decisions. In finance, a tree can represent a decision process, a rule system or the logic learned by a decision-tree model.

In [ ]:
approval_tree = {
    'question': 'Is annual income above 40k?',
    'yes': {'question': 'Any missed payments in last 12 months?', 'yes': 'manual review', 'no': 'likely approve'},
    'no': {'question': 'Is debt burden low?', 'yes': 'manual review', 'no': 'likely reject'}
}
approval_tree

## Section 5 — Practical finance takeaway

For most of this course, we work with **tables**, because the main use cases are customer-level and modelling-table problems. But it is useful to know that other structures exist and are more natural for other tasks.

### Notebook connection
- NB04 uses the **table** structure directly to build the modelling table.
- NB05b shows how **tree-based models** learn split logic from a table.

### Key ideas
1. Data structure is part of problem framing.
2. Tables remain the core structure for this course.
3. Long vs. wide changes what analysis is easy.
4. Graphs and trees help us think beyond flat spreadsheets.

---

*This is a compact-extension notebook. It supports manual section 2.2.2.2 and complements the core modelling pathway without being required for it.*